## Cell 1: 安装依赖 + Mount Drive

安装 `rfdetr` 及其训练/推理依赖（`--no-deps` 避免升级 Colab 预装的 `torch` / `torchvision`），显式补装 wheel 中声明的主要依赖，然后挂载 Google Drive。

In [ ]:
# Install RF-DETR and only the missing runtime deps, while avoiding broad Colab package upgrades.
!pip install -q "rfdetr==1.1.0" --no-deps
!pip install -q \
  cython \
  pycocotools \
  fairscale \
  scipy \
  timm \
  tqdm \
  accelerate \
  "transformers>=4.44,<5" \
  peft \
  ninja \
  einops \
  pylabel \
  rf100vl \
  supervision \
  matplotlib \
  pyDeprecate \
  open_clip_torch \
  onnx \
  onnxsim \
  onnx_graphsurgeon \
  polygraphy

# Fail early here if a required runtime dependency is still missing.
import torch
import torchvision
import transformers
import supervision as sv
from rfdetr import RFDETRBase

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("supervision:", sv.__version__)
print("RFDETR import OK")

from google.colab import drive
drive.mount("/content/drive")

## Cell 2: 配置路径变量

定义 Drive 上的数据集路径、Colab 本地缓存路径和模型输出目录，确保输出目录存在。

In [ ]:
from pathlib import Path

DRIVE_DIR            = Path("/content/drive/MyDrive/TreeLearn")
DRIVE_TDUS_DATA_DIR  = DRIVE_DIR / "tdus_resized"     # source dataset on Drive
LOCAL_WORK_DIR       = Path("/content/TreeLearn")
TDUS_DATA_DIR        = LOCAL_WORK_DIR / "tdus_resized" # local dataset copy for training
OUTPUT_DIR           = DRIVE_DIR / "rf_detr_tdus"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = OUTPUT_DIR / "best_checkpoint.pth"

print(f"DRIVE_TDUS_DATA_DIR : {DRIVE_TDUS_DATA_DIR}")
print(f"LOCAL TDUS_DATA_DIR : {TDUS_DATA_DIR}")
print(f"OUTPUT_DIR          : {OUTPUT_DIR}")
print(f"Drive dataset exists: {DRIVE_TDUS_DATA_DIR.exists()}")

## Cell 3: 复制数据集到 Colab 本地磁盘

将数据集从 Google Drive 复制到 `/content` 下的本地目录，减少训练时对 Drive FUSE 的频繁读取。若本地副本已存在则跳过。

In [ ]:
import shutil

assert DRIVE_TDUS_DATA_DIR.exists(), f"Drive dataset not found: {DRIVE_TDUS_DATA_DIR}"

if TDUS_DATA_DIR.exists():
    print(f"Local dataset already exists, skipping copy: {TDUS_DATA_DIR}")
else:
    print(f"Copying dataset from {DRIVE_TDUS_DATA_DIR} to {TDUS_DATA_DIR} ...")
    shutil.copytree(DRIVE_TDUS_DATA_DIR, TDUS_DATA_DIR)
    print("Dataset copy complete.")

print(f"Local data.yaml exists: {(TDUS_DATA_DIR / 'data.yaml').exists()}")

## Cell 4: 验证数据集格式

检查 `data.yaml` 存在、`nc` 与 `names` 字段正确，并统计 train / val 图像数量，确保有效训练集图像数 ≥ 2800。

In [ ]:
import yaml
from pathlib import Path

data_yaml = TDUS_DATA_DIR / "data.yaml"
assert data_yaml.exists(), f"data.yaml not found: {data_yaml}"
cfg = yaml.safe_load(data_yaml.read_text())
print("nc:", cfg["nc"])
print("names:", cfg["names"])

train_imgs = list((TDUS_DATA_DIR / "train" / "img").glob("*.jpg"))
val_imgs   = list((TDUS_DATA_DIR / "val"   / "img").glob("*.jpg"))
print(f"train: {len(train_imgs)} images")
print(f"val:   {len(val_imgs)} images")
assert len(train_imgs) >= 2800, f"Too few train images: {len(train_imgs)}"

## Cell 5: RF-DETR Fine-tune（核心训练）

加载 RF-DETR 预训练权重，在 TDUS 数据集上 fine-tune 50 个 epoch。`lr_encoder=1e-5` 保护 DINOv2 backbone 避免过拟合，`grad_accumulation_steps=4` 使有效 batch size = 16。预计 Colab L4 耗时约 30–60 分钟。

In [ ]:
from pathlib import Path
from rfdetr import RFDETRBase
from PIL import Image
import json
import os
import torch
import urllib.request
import yaml

RF_CACHE_DIR = Path("/content/rfdetr-cache")
RF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PRETRAIN_WEIGHTS = RF_CACHE_DIR / "rf-detr-base.pth"
PRETRAIN_URL = "https://storage.googleapis.com/rfdetr/rf-detr-base-coco.pth"

def ensure_pretrain_weights(weights_path: Path, url: str) -> None:
    needs_download = not weights_path.exists()

    if not needs_download:
        try:
            torch.load(weights_path, map_location="cpu", weights_only=False)
            print(f"Using existing weights: {weights_path}")
            return
        except Exception as e:
            print(f"Existing weights are unreadable, re-downloading: {e}")
            weights_path.unlink(missing_ok=True)

    print(f"Downloading pretrained weights to {weights_path} ...")
    urllib.request.urlretrieve(url, weights_path)
    torch.load(weights_path, map_location="cpu", weights_only=False)
    print("Pretrained weights ready.")

def convert_yolo_split_to_coco(dataset_dir: Path, split: str, class_names: list[str]) -> Path:
    split_dir = dataset_dir / split
    img_dir = split_dir / "img"
    label_dir = split_dir / "labels"
    out_json = split_dir / "_annotations.coco.json"

    image_paths = sorted(
        [*img_dir.glob("*.jpg"), *img_dir.glob("*.jpeg"), *img_dir.glob("*.png")]
    )
    assert image_paths, f"No images found in {img_dir}"

    images = []
    annotations = []
    categories = [{"id": i, "name": name} for i, name in enumerate(class_names)]
    ann_id = 1

    for image_id, image_path in enumerate(image_paths, start=1):
        with Image.open(image_path) as im:
            width, height = im.size

        images.append(
            {
                "id": image_id,
                "file_name": f"img/{image_path.name}",
                "width": width,
                "height": height,
            }
        )

        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue

        for line in label_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue

            class_id, x_center, y_center, box_w, box_h = map(float, line.split())
            class_id = int(class_id)

            abs_w = box_w * width
            abs_h = box_h * height
            abs_x = (x_center * width) - (abs_w / 2)
            abs_y = (y_center * height) - (abs_h / 2)

            annotations.append(
                {
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": class_id,
                    "bbox": [abs_x, abs_y, abs_w, abs_h],
                    "area": abs_w * abs_h,
                    "iscrowd": 0,
                }
            )
            ann_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }
    out_json.write_text(json.dumps(coco))
    print(f"Wrote {out_json} with {len(images)} images and {len(annotations)} annotations")
    return out_json

data_yaml = TDUS_DATA_DIR / "data.yaml"
cfg = yaml.safe_load(data_yaml.read_text())
class_names = cfg["names"]

convert_yolo_split_to_coco(TDUS_DATA_DIR, "train", class_names)
convert_yolo_split_to_coco(TDUS_DATA_DIR, "val", class_names)

# rfdetr 1.1.0 hardcodes "valid/" in build_roboflow(); our dataset uses "val/"
valid_dir = TDUS_DATA_DIR / "valid"
if not valid_dir.exists():
    valid_dir.symlink_to(TDUS_DATA_DIR / "val")
    print(f"Created symlink: valid/ -> val/")
else:
    print(f"valid/ already exists: {valid_dir}")

ensure_pretrain_weights(PRETRAIN_WEIGHTS, PRETRAIN_URL)

In [ ]:
os.environ["RF_HOME"] = str(RF_CACHE_DIR)
os.chdir(str(TDUS_DATA_DIR))

model = RFDETRBase(pretrain_weights=str(PRETRAIN_WEIGHTS))
model.train(
    dataset_dir=str(TDUS_DATA_DIR),
    epochs=50,
    batch_size=4,
    lr=1e-4,
    lr_encoder=1e-5,
    grad_accumulation_steps=4,
    warmup_epochs=3,
    weight_decay=1e-4,
    output_dir=str(OUTPUT_DIR),
    checkpoint_interval=10,
)
print("Training complete.")

## Cell 6: val 集检出率评估

加载 `best_checkpoint.pth`，对 val 集所有图像运行推理（threshold=0.3），统计检出率并将结果保存为 `val_detection_rate.json`。目标：检出率 ≥ 95%。

In [ ]:
import json
import numpy as np
from rfdetr import RFDETRBase

best_ckpt = OUTPUT_DIR / "best_checkpoint.pth"
assert best_ckpt.exists(), "best_checkpoint.pth not found — did training complete?"

model = RFDETRBase(pretrain_weights=str(best_ckpt))
val_imgs = sorted((TDUS_DATA_DIR / "val" / "img").glob("*.jpg"))

hit = 0
total = len(val_imgs)

for img_path in val_imgs:
    dets = model.predict(str(img_path), threshold=0.3)
    if len(dets.xyxy) > 0:
        hit += 1

detection_rate = hit / total
print(f"val 检出率: {hit}/{total} = {detection_rate:.1%}")

result = {"detection_rate": detection_rate, "n_hit": hit, "n_total": total}
(OUTPUT_DIR / "val_detection_rate.json").write_text(json.dumps(result, indent=2))
print("Saved val_detection_rate.json")

assert detection_rate >= 0.95, f"Detection rate {detection_rate:.1%} too low (need ≥95%)"

## Cell 7: 确认权重已保存至 Drive

列出 `OUTPUT_DIR` 下的所有产物，确认 `best_checkpoint.pth` 和 `checkpoint.pth` 均已写入 Drive。

In [ ]:
import shutil

# Verify key artifacts exist
for fname in ["best_checkpoint.pth", "checkpoint.pth"]:
    p = OUTPUT_DIR / fname
    if p.exists():
        print(f"✓ {fname} ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"✗ {fname} NOT FOUND")

print(f"\nAll artifacts in: {OUTPUT_DIR}")
print("Files:", [f.name for f in OUTPUT_DIR.iterdir()])